# 🦙 Llama-3.2-1B — LoRA Fine-Tuning (Framework-Free)

> **ရည်ရွယ်ချက်**: Framework (Axolotl, TRL, MS-Swift) တွေမသုံးပဲ **pure PyTorch + PEFT** နဲ့ LoRA fine-tuning လုပ်မယ်  
> **GPU**: RTX 3050 12GB | **Expected VRAM**: ~4-6 GB (LoRA ဖြစ်လို့ အသက်သာတယ်)

---

## 📋 Notebook Workflow

```
Step 1: Model Overview — Llama 3.2-1B ဘာတွေသိထားလဲ?
Step 2: Load & Inspect — Model Architecture ကို ကြည့်မယ်
Step 3: Probe Knowledge — ဘယ် domain တွေမှာ ကောင်းလဲ/ညံ့လဲ စစ်မယ်
Step 4: LoRA Setup — Framework မသုံးပဲ LoRA ကပ်မယ်
Step 5: Dataset Prep — Domain adaptation data ပြင်မယ်
Step 6: Training Loop — Pure PyTorch training loop ရေးမယ်
Step 7: Evaluate — Before vs After ယှဉ်မယ်
Step 8: Save & Merge — LoRA weights save/merge လုပ်မယ်
```

## 📖 Step 1: Llama 3.2-1B — Model Overview

### 🏗️ Architecture Details

| Property | Value |
|---|---|
| **Parameters** | 1.24B (1,236M) |
| **Architecture** | LlamaForCausalLM (Auto-regressive Transformer) |
| **Hidden Size** | 2048 |
| **Num Layers** | 16 |
| **Attention Heads** | 32 |
| **KV Heads** | 8 (Grouped-Query Attention) |
| **Intermediate Size** | 8192 |
| **Vocab Size** | 128,256 |
| **Context Length** | 128K tokens |
| **Precision** | BF16 |

### 📚 Training Data — ဘာတွေနဲ့ train ထားလဲ?

| Item | Detail |
|---|---|
| **Training Tokens** | Up to **9 Trillion tokens** |
| **Data Source** | Publicly available online data (web crawl, books, code, etc.) |
| **Data Cutoff** | December 2023 |
| **Creation Method** | Llama 3.1-8B ကနေ **Pruning** + Llama 3.1-8B/70B ကနေ **Knowledge Distillation** |
| **Post-training** | SFT → Rejection Sampling → DPO (multiple rounds) |

### 🌍 Supported Languages (Official)

`English` `German` `French` `Italian` `Portuguese` `Hindi` `Spanish` `Thai`

> ⚠️ **Myanmar language ကိုမပါဝင်ပါ** — ဒါကြောင့် Myanmar domain adaptation လုပ်ရင် LoRA အရမ်းအသုံးဝင်ပါတယ်

### 🎯 Model ဘာကောင်းလဲ / ဘာညံ့လဲ (1B Model ဖြစ်လို့)

| ✅ ကောင်းတာ | ❌ ညံ့တာ |
|---|---|
| Summarization | Complex reasoning |
| Instruction following | Math (GSM8K: 44.4%) |
| Text rewriting | Code generation (complex) |
| Simple Q&A | Non-supported languages |
| On-device deployment | Long-form analysis |
| Tool calling (basic) | Multi-step problem solving |

In [1]:
# ============================================================
# 📦 Step 2: Install Dependencies
# ============================================================
# pip install torch transformers peft datasets accelerate bitsandbytes

import subprocess, sys

packages = [
    "torch",
    "transformers>=4.43.0",
    "peft>=0.12.0",
    "datasets",
    "accelerate",
    "bitsandbytes",
]

for pkg in packages:
    print(f"✅ Checking: {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\n🎉 All packages ready!")

✅ Checking: torch
✅ Checking: transformers>=4.43.0
✅ Checking: peft>=0.12.0
✅ Checking: datasets
✅ Checking: accelerate
✅ Checking: bitsandbytes

🎉 All packages ready!


In [2]:
# ============================================================
# 📚 Step 3: Import Libraries
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
import json
import os
from pathlib import Path

# ============================================================
# 🔧 Device & Config
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"📊 Free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

# ============================================================
# 📁 Paths
# ============================================================
BASE_DIR = Path("/home/mr_cobot/Desktop/Git/ai/generative.ai/learning/fine_tuning/Llama-3.2-1B")
MODEL_ID = "meta-llama/Llama-3.2-1B"
OUTPUT_DIR = BASE_DIR / "lora_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"\n📁 Base Dir: {BASE_DIR}")
print(f"🤖 Model: {MODEL_ID}")
print(f"💾 Output: {OUTPUT_DIR}")

🖥️  Device: cuda
🎮 GPU: NVIDIA GeForce RTX 3060
💾 VRAM: 11.6 GB
📊 Free VRAM: 11.1 GB

📁 Base Dir: /home/mr_cobot/Desktop/Git/ai/generative.ai/learning/fine_tuning/Llama-3.2-1B
🤖 Model: meta-llama/Llama-3.2-1B
💾 Output: /home/mr_cobot/Desktop/Git/ai/generative.ai/learning/fine_tuning/Llama-3.2-1B/lora_output


## 🔍 Step 4: Load Model & Inspect Architecture

> Model ကို load လုပ်ပြီး architecture ကို inspect လုပ်မယ်  
> **LoRA ကပ်မယ့် layer တွေကို ဒီမှာ ရှာရမယ်**

### Llama 3.2-1B Transformer Block Structure
```
LlamaForCausalLM
├── model (LlamaModel)
│   ├── embed_tokens          ← Token Embedding (128256 × 2048)
│   ├── layers [0-15]         ← 16 Transformer Blocks
│   │   ├── self_attn         ← Self-Attention
│   │   │   ├── q_proj        ← Query Projection (2048 → 2048)    ✅ LoRA Target
│   │   │   ├── k_proj        ← Key Projection   (2048 → 512)     ✅ LoRA Target  
│   │   │   ├── v_proj        ← Value Projection  (2048 → 512)    ✅ LoRA Target
│   │   │   └── o_proj        ← Output Projection (2048 → 2048)   ✅ LoRA Target
│   │   ├── mlp               ← Feed-Forward Network
│   │   │   ├── gate_proj     ← Gate Projection   (2048 → 8192)   🔶 Optional LoRA
│   │   │   ├── up_proj       ← Up Projection     (2048 → 8192)   🔶 Optional LoRA
│   │   │   └── down_proj     ← Down Projection   (8192 → 2048)   🔶 Optional LoRA
│   │   ├── input_layernorm   ← RMSNorm (not trained)
│   │   └── post_attention_layernorm ← RMSNorm (not trained)
│   └── norm                  ← Final RMSNorm
└── lm_head                   ← Output Head (2048 → 128256)
```

> **GQA Note**: `num_attention_heads=32`, `num_key_value_heads=8`  
> Q has 32 heads, K/V has 8 heads → 4 Q heads share 1 KV head (memory efficient)

In [3]:
# ============================================================
# 🔍 Step 4a: Load Base Model (BF16)
# ============================================================
# RTX 3050 12GB ဆိုရင် BF16 load ရင် ~2.5GB VRAM သုံးမယ်
# QLoRA (4-bit) ဆိုရင် ~0.8GB ပဲသုံးမယ် — ပိုသက်သာတယ်

print("⏳ Loading base model... (ပထမဆုံးအကြိမ်ဆိုရင် download ရမယ်)")
print(f"   Model: {MODEL_ID}")

# --- Option A: BF16 (Normal) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,    # BF16 precision
    device_map="auto",             # Auto GPU/CPU split
    trust_remote_code=True,
)

# --- Option B: QLoRA (4-bit) — VRAM နည်းရင် ဒါသုံးပါ ---
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",           # NormalFloat4
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,      # Double quantization
# )
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )
# model = prepare_model_for_kbit_training(model)  # QLoRA prepare

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Llama 3.2 မှာ pad_token မရှိဘူး — eos_token ကို pad_token အဖြစ်သုံးမယ်
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id

print(f"\n✅ Model loaded!")
print(f"📊 Model dtype: {model.dtype}")
print(f"📝 Vocab size: {tokenizer.vocab_size}")
print(f"🔑 Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

# VRAM check
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 VRAM used: {allocated:.2f} GB")

⏳ Loading base model... (ပထမဆုံးအကြိမ်ဆိုရင် download ရမယ်)
   Model: meta-llama/Llama-3.2-1B


`torch_dtype` is deprecated! Use `dtype` instead!



✅ Model loaded!
📊 Model dtype: torch.bfloat16
📝 Vocab size: 128000
🔑 Pad token: '<|end_of_text|>' (id=128001)
💾 VRAM used: 2.30 GB


In [4]:
# ============================================================
# 🔍 Step 4b: Inspect Model Architecture (Deep Dive)
# ============================================================
# LoRA ကပ်ဖို့ ဘယ် layer တွေရှိလဲ သိဖို့ model structure ကြည့်ရမယ်

print("=" * 70)
print("🏗️  LLAMA 3.2-1B — FULL MODEL ARCHITECTURE")
print("=" * 70)

# Model config
config = model.config
print(f"\n📐 Architecture Config:")
print(f"   hidden_size:        {config.hidden_size}")
print(f"   num_hidden_layers:  {config.num_hidden_layers}")
print(f"   num_attention_heads:{config.num_attention_heads}")
print(f"   num_key_value_heads:{config.num_key_value_heads}")
print(f"   intermediate_size:  {config.intermediate_size}")
print(f"   vocab_size:         {config.vocab_size}")
print(f"   max_position_embed: {config.max_position_embeddings}")
print(f"   rms_norm_eps:       {config.rms_norm_eps}")
print(f"   rope_theta:         {config.rope_theta}")

# --- Layer-by-layer parameter count ---
print(f"\n{'=' * 70}")
print(f"📊 LAYER-BY-LAYER PARAMETER COUNT")
print(f"{'=' * 70}")

total_params = 0
layer_info = {}

for name, param in model.named_parameters():
    total_params += param.numel()
    # Group by layer type
    parts = name.split(".")
    if "layers" in parts:
        layer_num = parts[parts.index("layers") + 1]
        layer_type = ".".join(parts[parts.index("layers") + 2:])
        if layer_num == "0":  # Only show layer 0 as template
            shape_str = " × ".join(str(s) for s in param.shape)
            print(f"   layers.{layer_num}.{layer_type:40s} → [{shape_str}] = {param.numel():>12,}")
    elif "embed" in name or "lm_head" in name or "norm" in name:
        shape_str = " × ".join(str(s) for s in param.shape)
        print(f"   {name:50s} → [{shape_str}] = {param.numel():>12,}")

print(f"\n   {'─' * 55}")
print(f"   Total Parameters: {total_params:>15,}")
print(f"   Total Parameters: {total_params / 1e9:.3f}B")

# --- LoRA Target Candidates ---
print(f"\n{'=' * 70}")
print(f"🎯 LoRA TARGET CANDIDATES (Linear layers in each Transformer Block)")
print(f"{'=' * 70}")

# Find all unique linear layer names
linear_layers = set()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        # Get the layer type name
        parts = name.split(".")
        if "layers" in parts:
            layer_type = ".".join(parts[parts.index("layers") + 2:])
            in_f, out_f = module.in_features, module.out_features
            linear_layers.add((layer_type, in_f, out_f))

print(f"\n   {'Layer Name':<30s} {'In':>8s} → {'Out':>8s}  {'Params':>12s}  {'LoRA?'}")
print(f"   {'─' * 75}")
for layer_name, in_f, out_f in sorted(linear_layers):
    params = in_f * out_f
    is_lora = "✅ YES" if any(t in layer_name for t in ["q_proj", "k_proj", "v_proj", "o_proj"]) else "🔶 Optional"
    print(f"   {layer_name:<30s} {in_f:>8,} → {out_f:>8,}  {params:>12,}  {is_lora}")

🏗️  LLAMA 3.2-1B — FULL MODEL ARCHITECTURE

📐 Architecture Config:
   hidden_size:        2048
   num_hidden_layers:  16
   num_attention_heads:32
   num_key_value_heads:8
   intermediate_size:  8192
   vocab_size:         128256
   max_position_embed: 131072
   rms_norm_eps:       1e-05
   rope_theta:         500000.0

📊 LAYER-BY-LAYER PARAMETER COUNT
   model.embed_tokens.weight                          → [128256 × 2048] =  262,668,288
   layers.0.self_attn.q_proj.weight                  → [2048 × 2048] =    4,194,304
   layers.0.self_attn.k_proj.weight                  → [512 × 2048] =    1,048,576
   layers.0.self_attn.v_proj.weight                  → [512 × 2048] =    1,048,576
   layers.0.self_attn.o_proj.weight                  → [2048 × 2048] =    4,194,304
   layers.0.mlp.gate_proj.weight                     → [8192 × 2048] =   16,777,216
   layers.0.mlp.up_proj.weight                       → [8192 × 2048] =   16,777,216
   layers.0.mlp.down_proj.weight                     → [

## 🧪 Step 5: Probe Model Knowledge

> Model က ဘာတွေသိထားလဲ စစ်မယ်  
> Domain adaptation လုပ်ဖို့ — **မသိတဲ့ အပိုင်းကို ရှာရမယ်**

### Test Categories:
1. **General Knowledge** — အထွေထွေ အသိပညာ
2. **Myanmar Knowledge** — မြန်မာနိုင်ငံဆိုင်ရာ
3. **Technical/Domain** — Specific domain knowledge  
4. **Language** — Myanmar language capabilities
5. **Reasoning** — Logic & math

> 💡 **Tip**: Base model (non-instruct) ဖြစ်လို့ **text completion** style ဖြစ်မယ်  
> Question-Answer format မဟုတ်ပါ — text ကို ဆက်ရေးပေးတာပါ

In [5]:
# ============================================================
# 🧪 Step 5a: Knowledge Probing Function
# ============================================================

def probe_model(prompt, max_new_tokens=150, temperature=0.7, top_p=0.9):
    """
    Model ကို prompt ပေးပြီး generate လုပ်ခိုင်းတယ်
    
    Base model (non-instruct) ဖြစ်လို့ text completion style ပဲ ဖြစ်မယ်
    "The capital of France is" → "Paris, which is..." ဆိုပြီး ဆက်ရေးပေးမယ်
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.2,  # Repetition ရှောင်ဖို့
        )
    
    # Input tokens ဖယ်ပြီး generated tokens ပဲ decode မယ်
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    
    return response.strip()


def run_knowledge_test(category, tests):
    """
    Category တစ်ခုရဲ့ test prompts တွေကို run ပြီး results ပြတယ်
    """
    print(f"\n{'='*70}")
    print(f"🧪 CATEGORY: {category}")
    print(f"{'='*70}")
    
    results = []
    for i, (prompt, expected_topic) in enumerate(tests, 1):
        print(f"\n--- Test {i}: {expected_topic} ---")
        print(f"📝 Prompt: \"{prompt}\"")
        response = probe_model(prompt, max_new_tokens=100)
        print(f"🤖 Response: {response[:200]}...")
        results.append({
            "prompt": prompt,
            "topic": expected_topic,
            "response": response
        })
    
    return results

print("✅ Probe functions ready!")

✅ Probe functions ready!


In [6]:
# ============================================================
# 🧪 Step 5b: Test 1 — General Knowledge (English)
# ============================================================
# Model သိသင့်တာတွေ — 9T tokens နဲ့ train ထားလို့

general_tests = [
    ("The capital of France is", "Geography - France"),
    ("Python programming language was created by", "Tech - Python"),
    ("The theory of relativity was proposed by", "Science - Physics"),
    ("Machine learning is a subset of", "Tech - ML"),
    ("The largest planet in our solar system is", "Science - Astronomy"),
]

general_results = run_knowledge_test("General Knowledge (English)", general_tests)


🧪 CATEGORY: General Knowledge (English)

--- Test 1: Geography - France ---
📝 Prompt: "The capital of France is"
🤖 Response: Paris. The French are known for their love and respect towards the country's history, heritage as well as culture which makes them one of the most fascinating countries in Europe.
Paris offers many di...

--- Test 2: Tech - Python ---
📝 Prompt: "Python programming language was created by"
🤖 Response: Guido Van Rossum in the year 1989. Python is an open-source, general-purpose, high-level computer software and a powerful scripting programming language.
The term python means "bite-size" or small bit...

--- Test 3: Science - Physics ---
📝 Prompt: "The theory of relativity was proposed by"
🤖 Response: Albert Einstein, and his work on it has been a cornerstone for the modern physics. The Theory of Relativistic Quantum Mechanics is an attempt to apply this relativistically invariant framework in quan...

--- Test 4: Tech - ML ---
📝 Prompt: "Machine learning is a subs

In [7]:
# ============================================================
# 🧪 Step 5c: Test 2 — Myanmar Knowledge (Model မသိနိုင်တဲ့ အပိုင်း)
# ============================================================
# Myanmar language & knowledge — supported language မဟုတ်လို့ ညံ့မယ်

myanmar_tests = [
    ("The capital of Myanmar is", "Myanmar - Capital"),
    ("Myanmar is located in", "Myanmar - Geography"),
    ("Aung San Suu Kyi is known for", "Myanmar - Politics"),
    ("The Shwedagon Pagoda is located in", "Myanmar - Culture"),
    ("မြန်မာနိုင်ငံ၏ မြို့တော်မှာ", "Myanmar Language - Capital"),
    ("ရန်ကုန်မြို့သည်", "Myanmar Language - Yangon"),
]

myanmar_results = run_knowledge_test("Myanmar Knowledge", myanmar_tests)


🧪 CATEGORY: Myanmar Knowledge

--- Test 1: Myanmar - Capital ---
📝 Prompt: "The capital of Myanmar is"
🤖 Response: Yangon. The city has a population over 5 million people and it was once the seat for Burma’s military government, although they were forced to move out in May 2011.
Yangon is located on an isthmus tha...

--- Test 2: Myanmar - Geography ---
📝 Prompt: "Myanmar is located in"
🤖 Response: Southeast Asia and shares borders with Bangladesh, China (PRC), India, Laos, Thailand. Myanmar has a tropical climate that ranges from hot to humid. The average temperature of the country varies betwe...

--- Test 3: Myanmar - Politics ---
📝 Prompt: "Aung San Suu Kyi is known for"
🤖 Response: her advocacy work on human rights and democracy. In 1991, she was awarded the Nobel Peace Prize as part of a movement to end Myanmar’s military dictatorship.
In August this year Aung San Suu Kyi atten...

--- Test 4: Myanmar - Culture ---
📝 Prompt: "The Shwedagon Pagoda is located in"
🤖 Response: the h

In [8]:
# ============================================================
# 🧪 Step 5d: Test 3 — Domain-Specific (Your target domain)
# ============================================================
# ⬇️ ဒီမှာ သင် adapt လုပ်ချင်တဲ့ domain ကို test ပါ
# Example: Medical, Legal, Finance, Myanmar History, etc.

domain_tests = [
    # --- ဒီနေရာမှာ သင့် domain prompts ထည့်ပါ ---
    ("In traditional Myanmar medicine, thanaka is used for", "Myanmar Traditional Medicine"),
    ("The Irrawaddy River flows through", "Myanmar Geography Detail"),
    ("Mohinga is a traditional Myanmar dish made with", "Myanmar Food"),
    ("The Bagan temples were built during the", "Myanmar History"),
    ("Myanmar's currency is called", "Myanmar Economy"),
]

domain_results = run_knowledge_test("Domain-Specific Knowledge", domain_tests)


🧪 CATEGORY: Domain-Specific Knowledge

--- Test 1: Myanmar Traditional Medicine ---
📝 Prompt: "In traditional Myanmar medicine, thanaka is used for"
🤖 Response: skin care and as a facial mask. It’s also applied to wounds or burns.
Thanaka has been found effective in treating many health issues such as:
The word ‘thanak’ means ‘red’, which refers to the colour...

--- Test 2: Myanmar Geography Detail ---
📝 Prompt: "The Irrawaddy River flows through"
🤖 Response: Burma, Thailand and Laos. The waterway is the source of rice paddies for many Burmese people.
A large number of ethnic groups live in Myanmar; each has its own culture and traditions.
In Mandalay ther...

--- Test 3: Myanmar Food ---
📝 Prompt: "Mohinga is a traditional Myanmar dish made with"
🤖 Response: fermented fish paste called'mohingal'. It's usually served as breakfast, and has been eaten in Burma for centuries. The name means "fish soup" but it can also be described as an egg noodle soup that i...

--- Test 4: Myanmar His

In [9]:
# ============================================================
# 📊 Step 5e: Knowledge Gap Analysis — Summary
# ============================================================

print("=" * 70)
print("📊 KNOWLEDGE GAP ANALYSIS — SUMMARY")
print("=" * 70)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    KNOWLEDGE MAP                             │
├──────────────────────┬──────────────────────────────────────┤
│ ✅ STRONG (English)  │ General knowledge, Science, Tech     │
│                      │ Programming, World geography          │
├──────────────────────┼──────────────────────────────────────┤
│ 🔶 MEDIUM            │ Myanmar facts (in English)            │
│                      │ Basic cultural knowledge               │
├──────────────────────┼──────────────────────────────────────┤
│ ❌ WEAK              │ Myanmar language (not trained)         │
│                      │ Myanmar-specific domain knowledge      │
│                      │ Local context & nuances                │
└──────────────────────┴──────────────────────────────────────┘

🎯 DOMAIN ADAPTATION TARGETS:
   1. Myanmar language understanding/generation
   2. Myanmar cultural & historical knowledge
   3. Your specific domain (medical, legal, etc.)
   4. Instruction following in Myanmar context

💡 LoRA ကပ်ရမယ့် strategy:
   • target_modules: ["q_proj", "v_proj"] (minimum)
   • rank: 16-64 (domain adaptation အတွက် rank မြင့်ရမယ်)
   • Dataset: Domain-specific instruction-response pairs
""")

print("⬇️ Next: LoRA Setup!")

📊 KNOWLEDGE GAP ANALYSIS — SUMMARY

┌─────────────────────────────────────────────────────────────┐
│                    KNOWLEDGE MAP                             │
├──────────────────────┬──────────────────────────────────────┤
│ ✅ STRONG (English)  │ General knowledge, Science, Tech     │
│                      │ Programming, World geography          │
├──────────────────────┼──────────────────────────────────────┤
│ 🔶 MEDIUM            │ Myanmar facts (in English)            │
│                      │ Basic cultural knowledge               │
├──────────────────────┼──────────────────────────────────────┤
│ ❌ WEAK              │ Myanmar language (not trained)         │
│                      │ Myanmar-specific domain knowledge      │
│                      │ Local context & nuances                │
└──────────────────────┴──────────────────────────────────────┘

🎯 DOMAIN ADAPTATION TARGETS:
   1. Myanmar language understanding/generation
   2. Myanmar cultural & historical knowledge


## 🔧 Step 6: LoRA Setup — Framework-Free (PEFT Only)

### LoRA Math Recap
$$W' = W + \Delta W = W + A \times B$$

- $W$ = Original weight matrix (frozen ❄️)
- $A$ = Low-rank matrix ($d \times r$) — random init
- $B$ = Low-rank matrix ($r \times d$) — zero init  
- $r$ = rank (e.g., 16) — LoRA ရဲ့ capacity

### LoRA Hyperparameters ရှင်းလင်းချက်

| Parameter | Value | ရှင်းလင်းချက် |
|---|---|---|
| `r` (rank) | 16 | A,B matrix ရဲ့ rank — **မြင့်ရင် capacity များ, VRAM များ** |
| `lora_alpha` | 32 | Scaling factor — `alpha/r = 2` ဆို scaling=2x |
| `target_modules` | q,k,v,o_proj | LoRA ကပ်မယ့် layers |
| `lora_dropout` | 0.05 | Overfitting prevention |
| `bias` | "none" | Bias train မလုပ်ဘူး (memory save) |
| `task_type` | CAUSAL_LM | Text generation task |

### Scaling Formula
$$\text{output} = W \cdot x + \frac{\alpha}{r} \cdot (A \times B) \cdot x$$

> `alpha=32, r=16` ဆိုရင် scaling = $\frac{32}{16} = 2$ — LoRA contribution ကို 2x amplify လုပ်တယ်

In [10]:
# ============================================================
# 🔧 Step 6a: Define LoRA Configuration
# ============================================================
# ⚡ LoRA config ကို PEFT library ရဲ့ LoraConfig နဲ့ define လုပ်မယ်
# Framework (Axolotl, TRL) မသုံးပဲ — PEFT library ကိုပဲ တိုက်ရိုက်သုံးတယ်

lora_config = LoraConfig(
    # ─── Core LoRA Parameters ───
    r=16,                           # Rank — A,B matrix ရဲ့ dimension
                                    # r=8: lightweight, r=16: balanced, r=64: heavy
    
    lora_alpha=32,                  # Scaling factor — alpha/r = 32/16 = 2x
                                    # Rule: alpha = 2 * r (common practice)
    
    target_modules=[                # LoRA ကပ်မယ့် layers
        "q_proj",                   # Query projection   — attention pattern ပြောင်း
        "k_proj",                   # Key projection     — what to attend to
        "v_proj",                   # Value projection   — what info to extract
        "o_proj",                   # Output projection  — attention output ပြောင်း
        # --- Optional: MLP layers (domain adaptation အတွက် ထည့်နိုင်) ---
        # "gate_proj",              # MLP gate — knowledge storage
        # "up_proj",                # MLP up projection
        # "down_proj",              # MLP down projection
    ],
    
    lora_dropout=0.05,              # Dropout — overfitting prevention
    
    # ─── Task & Optimization ───
    bias="none",                    # "none": bias train မလုပ် (memory save)
                                    # "all": all biases train
                                    # "lora_only": LoRA layers ရဲ့ bias ပဲ
    
    task_type=TaskType.CAUSAL_LM,   # Causal Language Model (text generation)
)

# ============================================================
# 🔧 Step 6b: Apply LoRA to Model
# ============================================================
print("⏳ Applying LoRA adapters to model...")

# PEFT ရဲ့ get_peft_model() function ကို သုံးပြီး LoRA ကပ်မယ်
# ─── ဒီ function ကဘာလုပ်လဲ? ───
# 1. Base model ရဲ့ weight တွေကို FREEZE (requires_grad=False) 
# 2. Target modules တွေမှာ LoRA adapter (A, B matrices) ထပ်ထည့်
# 3. LoRA weights တွေပဲ trainable ဖြစ်အောင် set

model = get_peft_model(model, lora_config)

print("✅ LoRA applied!")
print(f"\n{'='*70}")
print(f"📊 TRAINABLE vs TOTAL PARAMETERS")
print(f"{'='*70}")

# ─── Parameter count comparison ───
model.print_trainable_parameters()
# Output: trainable params: X || all params: Y || trainable%: Z%

# ─── Detailed breakdown ───
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
frozen = total - trainable

print(f"\n   🔥 Trainable (LoRA):  {trainable:>12,} ({trainable/total*100:.4f}%)")
print(f"   ❄️  Frozen (Base):     {frozen:>12,} ({frozen/total*100:.4f}%)")
print(f"   📊 Total:             {total:>12,}")
print(f"\n   💾 LoRA weights size: ~{trainable * 2 / 1024**2:.1f} MB (BF16)")
print(f"   💡 Full model size:   ~{total * 2 / 1024**3:.2f} GB (BF16)")

# VRAM check
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"\n   🎮 Current VRAM: {allocated:.2f} GB")

⏳ Applying LoRA adapters to model...
✅ LoRA applied!

📊 TRAINABLE vs TOTAL PARAMETERS
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

   🔥 Trainable (LoRA):     3,407,872 (0.2750%)
   ❄️  Frozen (Base):     1,235,814,400 (99.7250%)
   📊 Total:             1,239,222,272

   💾 LoRA weights size: ~6.5 MB (BF16)
   💡 Full model size:   ~2.31 GB (BF16)

   🎮 Current VRAM: 2.32 GB


In [11]:
# ============================================================
# 🔍 Step 6c: Inspect LoRA Layers — ဘာတွေပြောင်းသွားလဲ ကြည့်မယ်
# ============================================================

print("=" * 70)
print("🔍 LoRA LAYERS INSPECTION — LoRA ကပ်ပြီးနောက် Model Structure")
print("=" * 70)

# Layer 0 ကို example အဖြစ် inspect လုပ်မယ်
print("\n📦 Layer 0 — Self Attention (LoRA applied):")
print("-" * 50)

layer0 = model.base_model.model.model.layers[0]

# Q projection ကို ကြည့်ရအောင်
q_proj = layer0.self_attn.q_proj
print(f"\n🎯 q_proj structure:")
print(f"   Type: {type(q_proj).__name__}")
print(f"   Original weight: {q_proj.base_layer.weight.shape} (FROZEN ❄️)")
print(f"   LoRA A: {q_proj.lora_A['default'].weight.shape}  (trainable 🔥)")
print(f"   LoRA B: {q_proj.lora_B['default'].weight.shape}  (trainable 🔥)")
print(f"   Scaling: {q_proj.scaling['default']}")  # alpha/r

# ─── LoRA A,B matrix ရဲ့ init values စစ်ကြည့်မယ် ───
print(f"\n🔬 LoRA A init (random gaussian):")
print(f"   mean={q_proj.lora_A['default'].weight.data.mean():.6f}")
print(f"   std={q_proj.lora_A['default'].weight.data.std():.6f}")

print(f"\n🔬 LoRA B init (zeros → initial ΔW = 0):")
print(f"   mean={q_proj.lora_B['default'].weight.data.mean():.6f}")
print(f"   std={q_proj.lora_B['default'].weight.data.std():.6f}")
print(f"   all_zeros={torch.all(q_proj.lora_B['default'].weight.data == 0).item()}")

# ─── LoRA ကပ်ထားတဲ့ layer list ───
print(f"\n{'='*70}")
print(f"📋 ALL LoRA LAYERS:")
print(f"{'='*70}")
for name, module in model.named_modules():
    if "lora_A" in name and "weight" not in name:
        layer_name = name.replace(".lora_A.default", "")
        print(f"   ✅ {layer_name}")

🔍 LoRA LAYERS INSPECTION — LoRA ကပ်ပြီးနောက် Model Structure

📦 Layer 0 — Self Attention (LoRA applied):
--------------------------------------------------

🎯 q_proj structure:
   Type: Linear
   Original weight: torch.Size([2048, 2048]) (FROZEN ❄️)
   LoRA A: torch.Size([16, 2048])  (trainable 🔥)
   LoRA B: torch.Size([2048, 16])  (trainable 🔥)
   Scaling: 2.0

🔬 LoRA A init (random gaussian):
   mean=0.000034
   std=0.012804

🔬 LoRA B init (zeros → initial ΔW = 0):
   mean=0.000000
   std=0.000000
   all_zeros=True

📋 ALL LoRA LAYERS:
   ✅ base_model.model.model.layers.0.self_attn.q_proj.lora_A
   ✅ base_model.model.model.layers.0.self_attn.q_proj
   ✅ base_model.model.model.layers.0.self_attn.k_proj.lora_A
   ✅ base_model.model.model.layers.0.self_attn.k_proj
   ✅ base_model.model.model.layers.0.self_attn.v_proj.lora_A
   ✅ base_model.model.model.layers.0.self_attn.v_proj
   ✅ base_model.model.model.layers.0.self_attn.o_proj.lora_A
   ✅ base_model.model.model.layers.0.self_attn.o_pr

## 📊 Step 7: Dataset Preparation — Domain Adaptation

### Dataset Format (Alpaca Style)
```json
{
    "instruction": "ဘာလုပ်ရမလဲ ညွှန်ကြားချက်",
    "input": "Optional — additional context",
    "output": "Model ပြန်ပေးရမယ့် အဖြေ"
}
```

### Training Data Pipeline
```
Raw Data → Alpaca Format → Tokenize → DataLoader → Training Loop
                ↓
         instruction + input + output
                ↓
         "[INST] {instruction}\n{input} [/INST] {output}"
                ↓
         Token IDs + Attention Mask + Labels
```

> ⚠️ **Important**: `labels` မှာ instruction/input part ကို `-100` (ignore) ထားမယ်  
> Output part ကိုပဲ loss calculate လုပ်မယ် — ဒါကို **causal LM loss masking** လို့ ခေါ်တယ်

In [14]:
# ============================================================
# 📊 Step 7a: Load & Format Dataset
# ============================================================

# ─── Option 1: Local JSONL file (train.jsonl) ───
def load_jsonl(filepath):
    """JSONL file ကို load လုပ်"""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# train.jsonl ကို load
train_data = load_jsonl(BASE_DIR / "train.jsonl")
print(f"📁 Loaded {len(train_data)} examples from train.jsonl")
print(f"\n📝 Example:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

# ─── Option 2: HuggingFace Dataset (ပိုများတဲ့ data) ───
# from datasets import load_dataset
# dataset = load_dataset("tatsu-lab/alpaca")  # 52K instruction examples
# train_data = dataset["train"]

print(f"\n💡 Production tip: tatsu-lab/alpaca (52K) သို့မဟုတ် custom data သုံးပါ")
print(f"   ယခု demo ဖြစ်လို့ {len(train_data)} examples ပဲ သုံးပါမယ်")

📁 Loaded 10 examples from train.jsonl

📝 Example:
{
  "instruction": "What is the capital of Myanmar?",
  "input": "",
  "output": "The capital of Myanmar is Naypyidaw. It has been the capital since 2006, replacing Yangon (Rangoon) as the administrative center."
}

💡 Production tip: tatsu-lab/alpaca (52K) သို့မဟုတ် custom data သုံးပါ
   ယခု demo ဖြစ်လို့ 10 examples ပဲ သုံးပါမယ်


In [15]:
# ============================================================
# 📊 Step 7b: Custom Dataset Class — Tokenization & Label Masking
# ============================================================
# Framework မသုံးပဲ manual tokenization လုပ်ရမယ်

class AlpacaDataset(Dataset):
    """
    Alpaca format dataset ကို tokenize လုပ်ပြီး PyTorch Dataset အဖြစ် ပြောင်းတယ်
    
    Format: "[INST] {instruction}\n{input} [/INST]\n{output}"
    
    Key Concept — Label Masking:
    ─────────────────────────────
    tokens:  [INST] instruction input [/INST] output <eos>
    labels:  -100   -100         -100  -100    output <eos>
                                                ↑ ဒီ part ကိုပဲ loss calculate
    
    -100 = CrossEntropyLoss က ignore လုပ်မယ့် special value
    """
    
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        instruction = item["instruction"]
        input_text = item.get("input", "")
        output = item["output"]
        
        # ─── Prompt Construction ───
        if input_text:
            prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
        else:
            prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
        
        full_text = prompt + output + self.tokenizer.eos_token
        
        # ─── Tokenize Full Text ───
        full_tokenized = self.tokenizer(
            full_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        
        input_ids = full_tokenized["input_ids"].squeeze()        # [max_length]
        attention_mask = full_tokenized["attention_mask"].squeeze()  # [max_length]
        
        # ─── Label Masking — Prompt part ကို ignore ───
        # Prompt ကိုပဲ tokenize ပြီး length ရယူ
        prompt_tokenized = self.tokenizer(
            prompt,
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )
        prompt_length = prompt_tokenized["input_ids"].shape[1]
        
        # Labels: prompt part ကို -100, output part ကို actual token ids
        labels = input_ids.clone()
        labels[:prompt_length] = -100  # Prompt part ignore
        
        # Padding tokens ကိုလည်း ignore
        labels[attention_mask == 0] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# ─── Dataset & DataLoader Create ───
MAX_SEQ_LEN = 512  # RTX 3050 12GB → 512 safe, 1024 ဆိုရင် VRAM tight

train_dataset = AlpacaDataset(train_data, tokenizer, max_length=MAX_SEQ_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,            # micro_batch_size = 1 (VRAM save)
    shuffle=True,
    drop_last=True,
)

print(f"✅ Dataset ready!")
print(f"   📊 Total examples: {len(train_dataset)}")
print(f"   📏 Max sequence length: {MAX_SEQ_LEN}")
print(f"   📦 Batch size: 1")
print(f"   🔄 Gradient accumulation: 4 (effective batch = 4)")

# ─── Inspect one example ───
sample = train_dataset[0]
print(f"\n🔍 Sample tokenization:")
print(f"   input_ids shape: {sample['input_ids'].shape}")
print(f"   labels shape:    {sample['labels'].shape}")
print(f"   Non-masked labels: {(sample['labels'] != -100).sum().item()} tokens")
print(f"   Masked (prompt):   {(sample['labels'] == -100).sum().item()} tokens")

# Show the decoded prompt vs response boundary
non_masked = sample["input_ids"][sample["labels"] != -100]
print(f"\n📝 Model will learn to generate:")
print(f"   \"{tokenizer.decode(non_masked[:50])}...\"")  # First 50 tokens of response

✅ Dataset ready!
   📊 Total examples: 10
   📏 Max sequence length: 512
   📦 Batch size: 1
   🔄 Gradient accumulation: 4 (effective batch = 4)

🔍 Sample tokenization:
   input_ids shape: torch.Size([512])
   labels shape:    torch.Size([512])
   Non-masked labels: 34 tokens
   Masked (prompt):   478 tokens

📝 Model will learn to generate:
   "The capital of Myanmar is Naypyidaw. It has been the capital since 2006, replacing Yangon (Rangoon) as the administrative center.<|end_of_text|>..."


## 🏋️ Step 8: Pure PyTorch Training Loop

> Framework (Trainer, SFTTrainer) မသုံးပဲ **raw training loop** ရေးမယ်  
> Code တိုင်းကို ရှင်းပြထားပါတယ်

### Training Loop Workflow
```
for each epoch:
    for each batch:
        1. Forward pass  → model(input_ids, labels) → loss
        2. Loss / grad_accum_steps  → gradient accumulation
        3. loss.backward()  → backprop (LoRA weights only)
        4. Every N steps: optimizer.step() → update LoRA weights
        5. scheduler.step() → learning rate adjust
        6. Log loss, VRAM
```

### Key Concepts:
- **Gradient Accumulation**: batch_size=1 ပေမယ့် N batches accumulate ပြီးမှ update → effective batch = N
- **Mixed Precision**: BF16 compute → faster & less VRAM
- **Gradient Checkpointing**: Activation memory save → VRAM ~40% save

In [16]:
# ============================================================
# 🏋️ Step 8a: Training Setup — Optimizer, Scheduler, Config
# ============================================================

# ─── Training Hyperparameters ───
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4              # LoRA learning rate — FFT ထက် မြင့်ရ (1e-5 vs 2e-4)
GRAD_ACCUM_STEPS = 4              # Gradient accumulation steps
WARMUP_STEPS = 10                 # Learning rate warmup
WEIGHT_DECAY = 0.01               # L2 regularization
MAX_GRAD_NORM = 1.0               # Gradient clipping

# ─── Enable Gradient Checkpointing ───
# VRAM save ဖို့ — activation values ကို recompute လုပ်တယ် (speed vs memory tradeoff)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()  # LoRA + gradient checkpointing compatibility

# ─── Optimizer ───
# AdamW — LoRA weights တွေကိုပဲ optimize လုပ်မယ်
# filter(lambda p: p.requires_grad, ...) → trainable params ပဲ ယူ
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),      # Adam default
    eps=1e-8,
)

# ─── Learning Rate Scheduler ───
# Cosine Annealing — slowly decay learning rate
total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM_STEPS

from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR

# Linear warmup + cosine decay
def lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    progress = float(current_step - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
    return max(0.0, 0.5 * (1.0 + __import__('math').cos(__import__('math').pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

print(f"🏋️ Training Config:")
print(f"   Epochs:           {NUM_EPOCHS}")
print(f"   Learning Rate:    {LEARNING_RATE}")
print(f"   Grad Accum Steps: {GRAD_ACCUM_STEPS}")
print(f"   Effective Batch:  {1 * GRAD_ACCUM_STEPS}")
print(f"   Total Steps:      {total_steps}")
print(f"   Warmup Steps:     {WARMUP_STEPS}")
print(f"   Weight Decay:     {WEIGHT_DECAY}")
print(f"   Max Grad Norm:    {MAX_GRAD_NORM}")
print(f"   Grad Checkpoint:  ✅ Enabled")

# VRAM check before training
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\n💾 VRAM before training:")
    print(f"   Allocated: {allocated:.2f} GB")
    print(f"   Reserved:  {reserved:.2f} GB")

🏋️ Training Config:
   Epochs:           3
   Learning Rate:    0.0002
   Grad Accum Steps: 4
   Effective Batch:  4
   Total Steps:      7
   Warmup Steps:     10
   Weight Decay:     0.01
   Max Grad Norm:    1.0
   Grad Checkpoint:  ✅ Enabled

💾 VRAM before training:
   Allocated: 2.32 GB
   Reserved:  2.33 GB


In [17]:
# ============================================================
# 🏋️ Step 8b: THE TRAINING LOOP — Pure PyTorch
# ============================================================
# ⚡ ဒီ cell ကို run ရင် actual training start မယ်

import time
from collections import defaultdict

print("=" * 70)
print("🏋️ STARTING LoRA TRAINING — Pure PyTorch")
print("=" * 70)

# Training state tracking
training_log = []
global_step = 0
best_loss = float("inf")

model.train()  # Training mode ON

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    epoch_start = time.time()
    accumulated_loss = 0.0
    
    print(f"\n{'─'*70}")
    print(f"📅 Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"{'─'*70}")
    
    for step, batch in enumerate(train_loader):
        # ─── Step 1: Move batch to device ───
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # ─── Step 2: Forward Pass ───
        # model() calls:
        #   1. embed_tokens(input_ids)        → embeddings
        #   2. layers[0-15] forward           → hidden states
        #      - self_attn (with LoRA!)       → attention output  
        #      - mlp                          → FFN output
        #   3. lm_head(hidden_states)         → logits
        #   4. CrossEntropyLoss(logits, labels) → loss
        #      (labels=-100 positions are ignored!)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        
        loss = outputs.loss
        
        # ─── Step 3: Gradient Accumulation ───
        # batch_size=1 ပေမယ့် 4 batches ရဲ့ gradient ကို accumulate
        # Effective batch size = 1 × 4 = 4
        loss = loss / GRAD_ACCUM_STEPS
        accumulated_loss += loss.item()
        
        # ─── Step 4: Backward Pass ───
        # gradient calculate — LoRA weights (A, B) ဆီပဲ gradient flow မယ်
        # Base model weights ← requires_grad=False ဖြစ်လို့ gradient compute မလုပ်
        loss.backward()
        
        # ─── Step 5: Parameter Update (every N steps) ───
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            # Gradient clipping — exploding gradients prevention
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                MAX_GRAD_NORM,
            )
            
            # Update LoRA weights
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            
            # ─── Logging ───
            current_lr = scheduler.get_last_lr()[0]
            
            log_entry = {
                "epoch": epoch + 1,
                "global_step": global_step,
                "loss": accumulated_loss,
                "lr": current_lr,
            }
            training_log.append(log_entry)
            
            # Print progress
            print(
                f"  Step {global_step:4d} | "
                f"Loss: {accumulated_loss:.4f} | "
                f"LR: {current_lr:.2e} | "
                f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB"
                if torch.cuda.is_available() else
                f"  Step {global_step:4d} | "
                f"Loss: {accumulated_loss:.4f} | "
                f"LR: {current_lr:.2e}"
            )
            
            epoch_loss += accumulated_loss
            accumulated_loss = 0.0
            
            # Save best model
            if log_entry["loss"] < best_loss:
                best_loss = log_entry["loss"]
    
    # Epoch summary
    epoch_time = time.time() - epoch_start
    avg_loss = epoch_loss / max(1, global_step)
    print(f"\n  📊 Epoch {epoch+1} Summary:")
    print(f"     Avg Loss: {avg_loss:.4f}")
    print(f"     Time: {epoch_time:.1f}s")
    print(f"     Best Loss: {best_loss:.4f}")

print(f"\n{'='*70}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Total steps: {global_step}")
print(f"   Best loss: {best_loss:.4f}")
print(f"{'='*70}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


🏋️ STARTING LoRA TRAINING — Pure PyTorch

──────────────────────────────────────────────────────────────────────
📅 Epoch 1/3
──────────────────────────────────────────────────────────────────────
  Step    1 | Loss: 1.8112 | LR: 2.00e-05 | VRAM: 2.5GB
  Step    2 | Loss: 1.4830 | LR: 4.00e-05 | VRAM: 2.5GB

  📊 Epoch 1 Summary:
     Avg Loss: 1.6471
     Time: 2.7s
     Best Loss: 1.4830

──────────────────────────────────────────────────────────────────────
📅 Epoch 2/3
──────────────────────────────────────────────────────────────────────
  Step    3 | Loss: 1.4677 | LR: 6.00e-05 | VRAM: 2.5GB
  Step    4 | Loss: 2.0400 | LR: 8.00e-05 | VRAM: 2.5GB

  📊 Epoch 2 Summary:
     Avg Loss: 0.8769
     Time: 2.6s
     Best Loss: 1.4677

──────────────────────────────────────────────────────────────────────
📅 Epoch 3/3
──────────────────────────────────────────────────────────────────────
  Step    5 | Loss: 1.2580 | LR: 1.00e-04 | VRAM: 2.5GB
  Step    6 | Loss: 1.9030 | LR: 1.20e-04 | VRAM

In [18]:
# ============================================================
# 📈 Step 8c: Training Loss Visualization
# ============================================================

if training_log:
    print("📈 TRAINING LOSS CURVE")
    print("=" * 50)
    
    # Simple ASCII loss chart
    losses = [entry["loss"] for entry in training_log]
    max_loss = max(losses) if losses else 1
    min_loss = min(losses) if losses else 0
    chart_width = 40
    
    for i, entry in enumerate(training_log):
        loss = entry["loss"]
        bar_len = int((loss - min_loss) / (max_loss - min_loss + 1e-8) * chart_width)
        bar = "█" * max(1, bar_len)
        print(f"  Step {entry['global_step']:3d} | {bar} {loss:.4f}")
    
    print(f"\n  📊 Loss range: {min_loss:.4f} → {losses[-1]:.4f}")
    if len(losses) > 1:
        improvement = (losses[0] - losses[-1]) / losses[0] * 100
        print(f"  📉 Improvement: {improvement:.1f}%")
else:
    print("⚠️ No training logs yet — run the training cell first")

📈 TRAINING LOSS CURVE
  Step   1 | ██████ 1.3120
  Step   2 | ███████████████ 1.5305
  Step   3 | ███████████████████████████████████████ 2.1401
  Step   4 | █ 1.1386
  Step   5 | ████ 1.2443
  Step   6 | ██████████████ 1.5133

  📊 Loss range: 1.1386 → 1.5133
  📉 Improvement: -15.3%


## 🧪 Step 9: Evaluation — Before vs After Comparison

> Training ပြီးရင် — model ဘယ်လောက်တိုးတက်သွားလဲ စစ်မယ်  
> **Same prompts** ကိုပဲ LoRA model နဲ့ ပြန် test ပြီး compare မယ်

In [19]:
# ============================================================
# 🧪 Step 9a: Evaluate LoRA-tuned Model
# ============================================================

model.eval()  # Evaluation mode — dropout OFF

print("=" * 70)
print("🧪 POST-TRAINING EVALUATION")
print("=" * 70)

# ─── Same test prompts as before ───
eval_prompts = [
    ("What is the capital of Myanmar?", "Myanmar Capital"),
    ("Explain what machine learning is in simple terms.", "ML Explanation"),
    ("The Shwedagon Pagoda is located in", "Myanmar Culture"),
    ("Write a Python function to calculate factorial.", "Code Gen"),
    ("Mohinga is a traditional Myanmar dish made with", "Myanmar Food"),
]

print("\n🤖 LoRA-tuned Model Responses:")
print("-" * 50)

for prompt, topic in eval_prompts:
    print(f"\n--- {topic} ---")
    print(f"📝 Prompt: \"{prompt}\"")
    
    # Generate with LoRA model
    response = probe_model(prompt, max_new_tokens=150)
    print(f"🤖 Response: {response[:300]}")

print(f"\n{'='*70}")
print("💡 Compare these results with Step 5 outputs!")
print("   Model quality depends on dataset size & domain relevance")
print("   10 examples = demo only — real training needs 1K-50K+ examples")

🧪 POST-TRAINING EVALUATION

🤖 LoRA-tuned Model Responses:
--------------------------------------------------

--- Myanmar Capital ---
📝 Prompt: "What is the capital of Myanmar?"
🤖 Response: ?
A. Yangon
B. Mandalay
C. Naypyidaw
D. Rangoon
Answer: C

--- ML Explanation ---
📝 Prompt: "Explain what machine learning is in simple terms."
🤖 Response: Give examples of ML applications.
Machine Learning (ML) describes a set of methods that can learn from data to make predictions without being explicitly programmed or taught how to do so by humans, which means we are able to use algorithms and processes designed for computers but applied to real-wor

--- Myanmar Culture ---
📝 Prompt: "The Shwedagon Pagoda is located in"
🤖 Response: downtown Yangon. It's the biggest pagoda and a symbol of Myanmar, Buddhism, culture.
It was built by King Bodawpaya around 1821 to celebrate his victory over the Burmese king Sagaing. This event also marked the beginning of construction of other large Buddhist temples 

In [20]:
# ============================================================
# 🧪 Step 9b: Perplexity Comparison (Optional — Quantitative Eval)
# ============================================================
# Perplexity = exp(loss) — နိမ့်ရင် ကောင်းတယ်

import math

model.eval()

print("=" * 70)
print("📊 PERPLEXITY EVALUATION (Lower = Better)")
print("=" * 70)

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        
        # Count non-masked tokens
        non_masked = (labels != -100).sum().item()
        total_loss += outputs.loss.item() * non_masked
        total_tokens += non_masked

avg_loss = total_loss / max(1, total_tokens)
perplexity = math.exp(avg_loss)

print(f"\n   Average Loss:  {avg_loss:.4f}")
print(f"   Perplexity:    {perplexity:.2f}")
print(f"   Total Tokens:  {total_tokens:,}")
print(f"\n   📈 Perplexity Guide:")
print(f"      < 10:   Excellent (well-learned)")
print(f"      10-50:  Good")
print(f"      50-100: Moderate")
print(f"      > 100:  Poor (needs more training/data)")

📊 PERPLEXITY EVALUATION (Lower = Better)

   Average Loss:  1.3379
   Perplexity:    3.81
   Total Tokens:  426

   📈 Perplexity Guide:
      < 10:   Excellent (well-learned)
      10-50:  Good
      50-100: Moderate
      > 100:  Poor (needs more training/data)


## 💾 Step 10: Save & Merge LoRA Weights

### Save Options

| Method | Size | Use Case |
|---|---|---|
| **LoRA Only** | ~5-20 MB | Adapter ပဲ save — base model ခွဲသုံးမယ် |
| **Merged** | ~2.5 GB | Base + LoRA merge — standalone model |

### LoRA Save/Load Workflow
```
Save LoRA adapter → lora_output/adapter_model.safetensors (~5MB)
                   + lora_output/adapter_config.json

Load for inference:
  1. Load base model (meta-llama/Llama-3.2-1B)
  2. Load LoRA adapter (PeftModel.from_pretrained)
  3. Ready! Base model + LoRA = Fine-tuned model

Merge (Optional):
  1. model.merge_and_unload()  → LoRA weights ကို base weights ထဲ ပေါင်း
  2. Save as standalone model  → LoRA adapter မလိုတော့ဘူး
```

In [21]:
# ============================================================
# 💾 Step 10a: Save LoRA Adapter Only (~5-20 MB)
# ============================================================

LORA_SAVE_PATH = OUTPUT_DIR / "lora_adapter"

print("💾 Saving LoRA adapter weights...")
model.save_pretrained(str(LORA_SAVE_PATH))
tokenizer.save_pretrained(str(LORA_SAVE_PATH))

# Check saved files
print(f"\n📁 Saved to: {LORA_SAVE_PATH}")
print(f"   Files:")
for f in sorted(LORA_SAVE_PATH.iterdir()):
    size = f.stat().st_size
    if size > 1024 * 1024:
        print(f"   📄 {f.name} ({size / 1024**2:.1f} MB)")
    elif size > 1024:
        print(f"   📄 {f.name} ({size / 1024:.1f} KB)")
    else:
        print(f"   📄 {f.name} ({size} B)")

print(f"\n✅ LoRA adapter saved!")
print(f"   💡 Base model ကိုတော့ '{MODEL_ID}' ကနေ HuggingFace ကနေ ပြန်ယူနိုင်")
print(f"   💡 LoRA adapter (~MB) ကို share/deploy လုပ်ရတာ လွယ်တယ်")

💾 Saving LoRA adapter weights...

📁 Saved to: /home/mr_cobot/Desktop/Git/ai/generative.ai/learning/fine_tuning/Llama-3.2-1B/lora_output/lora_adapter
   Files:
   📄 README.md (5.1 KB)
   📄 adapter_config.json (1005 B)
   📄 adapter_model.safetensors (13.0 MB)
   📄 special_tokens_map.json (335 B)
   📄 tokenizer.json (16.4 MB)
   📄 tokenizer_config.json (49.4 KB)

✅ LoRA adapter saved!
   💡 Base model ကိုတော့ 'meta-llama/Llama-3.2-1B' ကနေ HuggingFace ကနေ ပြန်ယူနိုင်
   💡 LoRA adapter (~MB) ကို share/deploy လုပ်ရတာ လွယ်တယ်


In [ ]:
# ============================================================
# 💾 Step 10b: Load LoRA Adapter for Inference (Later Use)
# ============================================================
# ─── LoRA adapter ကို နောက်မှ load ပြန်သုံးချင်ရင် ───

# # Step 1: Base model load
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
# 
# # Step 2: LoRA adapter load & attach
# model = PeftModel.from_pretrained(
#     base_model,
#     str(LORA_SAVE_PATH),  # LoRA adapter path
# )
# 
# # Step 3: Ready for inference!
# model.eval()

print("💡 LoRA adapter reload code (uncomment to use)")
print("   1. Load base model")
print("   2. PeftModel.from_pretrained(base_model, lora_path)")
print("   3. model.eval() → Ready!")

In [22]:
# ============================================================
# 💾 Step 10c: Merge LoRA into Base Model (Optional)
# ============================================================
# LoRA weights ကို base model ထဲ permanently merge လုပ်တယ်
# Merge ပြီးရင် standalone model ဖြစ်သွားတယ် — LoRA adapter မလိုတော့ဘူး

MERGED_SAVE_PATH = OUTPUT_DIR / "merged_model"

print("🔀 Merging LoRA weights into base model...")

# merge_and_unload():
# 1. W' = W + (alpha/r) * A × B  → LoRA weights ကို base weights ထဲ ပေါင်း
# 2. LoRA adapter ကို ဖယ်ထုတ် (unload)
# 3. Pure base model (with updated weights) ပြန်ရ
merged_model = model.merge_and_unload()

print("💾 Saving merged model...")
merged_model.save_pretrained(str(MERGED_SAVE_PATH))
tokenizer.save_pretrained(str(MERGED_SAVE_PATH))

# Check size
total_size = sum(f.stat().st_size for f in MERGED_SAVE_PATH.rglob("*") if f.is_file())
print(f"\n✅ Merged model saved!")
print(f"   📁 Path: {MERGED_SAVE_PATH}")
print(f"   📊 Size: {total_size / 1024**3:.2f} GB")
print(f"\n   💡 LoRA adapter: ~MB (lightweight)")
print(f"   💡 Merged model: ~GB (standalone, no adapter needed)")

🔀 Merging LoRA weights into base model...
💾 Saving merged model...

✅ Merged model saved!
   📁 Path: /home/mr_cobot/Desktop/Git/ai/generative.ai/learning/fine_tuning/Llama-3.2-1B/lora_output/merged_model
   📊 Size: 2.32 GB

   💡 LoRA adapter: ~MB (lightweight)
   💡 Merged model: ~GB (standalone, no adapter needed)


## 📋 Quick Reference — Full LoRA Pipeline (Framework-Free)

### Complete Workflow Diagram
```
┌──────────────────────────────────────────────────────────────────┐
│                    LoRA Fine-Tuning Pipeline                      │
├──────────────────────────────────────────────────────────────────┤
│                                                                   │
│  ① PROBE          ② SETUP          ③ TRAIN         ④ DEPLOY     │
│  ─────────        ────────         ──────────      ──────────    │
│  Load Base   →   Apply LoRA   →   Training    →   Save/Merge    │
│  Model            Config          Loop (PyTorch)   LoRA Weights  │
│  Test Knowledge   r, alpha        Forward Pass                    │
│  Find Gaps        targets         Backward Pass    LoRA Only     │
│                   dropout         Update LoRA      (~5-20 MB)    │
│                                   (base frozen)    or Merged     │
│                                                    (~2.5 GB)     │
│                                                                   │
├──────────────────────────────────────────────────────────────────┤
│  Key Code:                                                        │
│  ─────────                                                        │
│  model = AutoModelForCausalLM.from_pretrained(MODEL_ID)           │
│  config = LoraConfig(r=16, lora_alpha=32, target_modules=[...])   │
│  model = get_peft_model(model, config)                            │
│  for batch in loader: loss = model(**batch).loss; loss.backward() │
│  model.save_pretrained(path)     # LoRA only                     │
│  model.merge_and_unload()        # Merge into base               │
└──────────────────────────────────────────────────────────────────┘
```

### Framework Comparison

| Feature | This Notebook (Framework-Free) | Axolotl/TRL |
|---|---|---|
| **Code Visibility** | ✅ Every line visible | ❌ Abstracted away |
| **Learning Value** | ✅ Maximum understanding | 🔶 Config-based |
| **Flexibility** | ✅ Full control | 🔶 Config options |
| **Setup Speed** | ❌ Manual setup | ✅ 1 YAML file |
| **Best Practices** | ❌ DIY | ✅ Built-in |
| **Production** | 🔶 Needs hardening | ✅ Production-ready |

### 🎯 Next Steps for Domain Adaptation
1. **Larger Dataset**: 1K-50K+ domain-specific instruction pairs ပြင်ပါ
2. **MLP Layers**: `gate_proj`, `up_proj`, `down_proj` ကိုလည်း `target_modules` ထဲ ထည့်ပါ
3. **Higher Rank**: Domain adaptation အတွက် `r=32-64` သုံးပါ
4. **Evaluation**: Domain-specific eval dataset ဖန်တီးပါ
5. **QLoRA**: VRAM save ဖို့ 4-bit quantization + LoRA သုံးပါ